# Module 05 — AI Workbook Companion (GPU-friendly generator analysis)

A lightweight companion to
[generator_tutorial_gpu_annotated.ipynb](../generator_tutorial_gpu_annotated.ipynb).
It does **not** replace it — it adds the supervision layer from
[Module 11](../../11/README.md) to Module 05's topic: **reducing large
per-event quantities from a physics generator**, GPU-style.

New here? Read [Module 11's README](../../11/README.md) first — it explains the
5-phase loop and the known ways AI gets parallel code wrong. This module focuses
on one of them: **a silent dtype/precision change**. GPUs default to `float32`,
and an AI porting an analysis to the GPU often quietly accumulates a large sum in
`float32`, losing precision.

## The loop, applied to a weighted sum

Your task: sum millions of event weights (e.g. a total cross-section or an
integral) the way a GPU reduction would. You may have an AI write it. Your job is
everything around it.

1. **SPECIFY** — Contract: input dtype, the accumulation dtype, the acceptable
   relative error, and the metric.
2. **GENERATE** — Ask the AI for the reduction. Keep it in its own file.
3. **PREDICT** — *Before running:* a naive `float32` accumulation of many small
   numbers loses low-order bits as the running total grows. Do you see that risk
   in the generated code?
4. **VERIFY** — Use [verify_weighted_sum.py](./verify_weighted_sum.py): a
   `float64` (and exact `math.fsum`) reference with a relative-error PASS/FAIL gate.
5. **DIAGNOSE** — Quantify the error, and explain the fix (accumulate in
   `float64`, or use pairwise/Kahan summation — the same choices you'd make in a
   GPU reduction).

## The adversarial exercise

[adversarial_weighted_sum_buggy.py](./adversarial_weighted_sum_buggy.py) is a
reduction "an AI wrote for you." It runs and returns a plausible number, but it
accumulates in `float32`, so the total is wrong in a way that grows with the
number of events — exactly what happens when a GPU reduction uses the default
single precision without care.

Your job:

1. Predict the sign and rough size of the error *before* running.
2. Run it and compare to the `float64` / `fsum` reference.
3. State the exact bug and the fix.
4. Prove the fix within a stated relative tolerance.

Could an AI catch this? Sometimes — but "it returned a number close to what I
expected" passes a casual review while the precision is quietly degraded. A
relative-error check against a high-precision reference catches it. That is your
job.

A worked solution and explanation are in [solution.ipynb](./solution.ipynb). Try it
yourself first.

## Reflection prompt (write this down)

- Why does `float32` accumulation lose accuracy as the running sum grows large
  relative to each addend?
- How is this the same concern as a GPU reduction that accumulates in `float32`?
  What do pairwise or Kahan summation buy you?
- What relative tolerance is acceptable for your physics result, and how did you
  decide it before measuring?


In [ ]:
# Prerequisites: verify the GPU toolchain before running this workbook.
import pathlib, runpy
_here = pathlib.Path.cwd()
for _d in (_here, *_here.parents):
    if (_d / "check_gpu.py").exists():
        runpy.run_path(str(_d / "check_gpu.py"), run_name="__main__")
        break
else:
    print("check_gpu.py not found - run `python check_gpu.py` from the repo root.")

## Step 1 - Reproduce the bug

Run the problem program [adversarial_weighted_sum_buggy.py](./adversarial_weighted_sum_buggy.py). Read it first and predict what will happen before you run this cell.

In [ ]:
!python adversarial_weighted_sum_buggy.py

## Step 2 - Run your verification harness

[verify_weighted_sum.py](./verify_weighted_sum.py) ships a CPU reference and a PASS/FAIL gate, but it **defaults to FAIL** until you complete the function under test. Edit the file, then re-run this cell until it prints `PASS`.

In [ ]:
!python verify_weighted_sum.py

## Next

Once your harness prints `PASS`, open **[solution.ipynb](./solution.ipynb)** to compare your fix with the worked answer and explanation. See [Module 11](../../11/README.md) for the method behind this workbook.